# 2. BERTopic experiments and topic interpretation
**Questions:** Which configuration yields useful themes? How stable are its assignments?

Run notebook 01 first. Embeddings remain fixed across the bounded grid. Model selection
considers coherence, coverage, granularity, and article evidence together. Topic IDs have
meaning only within a run. HDBSCAN membership strength is not an LDA topic probability.

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/modern-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import CORPUS_DIR, audit_corpus, load_corpus, corpus_hash, digest, write_json
from shared import evaluation as ev
from src import topic_helpers as h, artifacts as a
OUT = ROOT/'Results/modern'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

df, embeddings = h.load_inputs(ROOT)
reservation = json.loads((OUT/'review_sample.json').read_text())
assert reservation['corpus_hash'] == corpus_hash(df)
texts, dictionary = ev.reference_texts(df)
# min_df refers to articles, not BERTopic's concatenated cluster documents.
vocabulary = CountVectorizer(stop_words='english', ngram_range=(1,1), min_df=5).fit(df.text.map(ev.representation_text)).vocabulary_
# Cache identity includes executable fitting code and helper implementations.
notebook = json.loads((ROOT/'notebooks/02_modern_topic_modeling.ipynb').read_text())
fit_source = ''.join(''.join(c['source']) for c in notebook['cells'] if 'model-fit' in c.get('metadata',{}).get('tags',[]))
implementation_hash = digest([fit_source] + [(ROOT/p).read_text() for p in ['src/artifacts.py','src/topic_helpers.py','../shared/evaluation.py']])

/Users/marykate/Desktop/F26/DataMiningProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Standard fitting path
UMAP reduces contextual embeddings to five dimensions; HDBSCAN finds dense groups and
keeps unassigned articles at −1; c-TF-IDF provides unigram descriptions.
We use [BERTopic's precomputed-embedding interface](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html)
with its real fitted UMAP and HDBSCAN components. No fixed-coordinate adapters or replacement cluster labels.
The combined fit includes reduction, clustering, and representation; evaluation is timed separately.

In [3]:
def fit_configuration(neighbors, cluster_size, seed=42):
    """One standard BERTopic fit; all experimental choices are visible here."""
    run_id = f'n{neighbors}_c{cluster_size}_s{seed}'
    folder = OUT/'runs'/run_id
    parameters = dict(n_neighbors=neighbors, min_cluster_size=cluster_size, seed=seed,
                      n_components=5, min_dist=0, umap_metric='cosine', min_samples=10,
                      cluster_metric='euclidean', cluster_selection_method='eom',
                      min_article_frequency=5, ngram_range=[1,1], top_n_words=10)
    identity = a.run_identity(df, embeddings, parameters, vocabulary, implementation_hash, reservation)
    if (folder/'manifest.json').exists():
        return a.load_run(folder, expected=identity)
    model = BERTopic(
        embedding_model=None,
        umap_model=UMAP(n_neighbors=neighbors, n_components=5, min_dist=0, metric='cosine', random_state=seed),
        hdbscan_model=HDBSCAN(min_cluster_size=cluster_size, min_samples=10, metric='euclidean',
                             cluster_selection_method='eom', prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words='english', ngram_range=(1,1), vocabulary=vocabulary),
        top_n_words=10, calculate_probabilities=False, verbose=False)
    print(f'{run_id}: fitting', flush=True)
    started = time.perf_counter()
    model.fit_transform(df.text.tolist(), embeddings=embeddings)
    fitting_seconds = time.perf_counter() - started
    started = time.perf_counter()
    metrics, per_topic = h.evaluate(model, embeddings, texts, dictionary)
    metrics.update(run_id=run_id, neighbors=neighbors, cluster_size=cluster_size, seed=seed,
                   fitting_representation_seconds=fitting_seconds,
                   evaluation_seconds=time.perf_counter()-started)
    a.export_run(model, df, embeddings, folder, run_id, set(reservation['article_ids']))
    per_topic.to_csv(folder/'coherence.csv', index=False)
    a.save_run_manifest(folder, identity, metrics)
    return model, metrics

In [4]:
grid_rows = []
for neighbors in [15,30]:
    for cluster_size in [25,50,100]:
        _, metrics = fit_configuration(neighbors, cluster_size)
        grid_rows.append({k:v for k,v in metrics.items() if k != 'topic_sizes'})
experiments = pd.DataFrame(grid_rows)
experiments.to_csv(OUT/'experiments.csv', index=False)
display(experiments[['run_id','topic_count','outlier_fraction','coherence','vocabulary_coverage','keyword_diversity','silhouette_cosine','fitting_representation_seconds']])

,run_id,topic_count,outlier_fraction,coherence,vocabulary_coverage,keyword_diversity,silhouette_cosine,fitting_representation_seconds
0,n15_c25_s42,76,0.202400,0.692091,1.0,0.643421,0.114374,13.588574
1,n15_c50_s42,40,0.199264,0.662209,1.0,0.622500,0.119224,7.269465
2,n15_c100_s42,21,0.146072,0.640805,1.0,0.585714,0.104999,8.030061
3,n30_c25_s42,70,0.180442,0.704696,1.0,0.651429,0.121705,10.049607
4,n30_c50_s42,35,0.190807,0.645542,1.0,0.600000,0.122660,9.120613
5,n30_c100_s42,24,0.222040,0.629542,1.0,0.608333,0.134029,8.978651


## Development evidence and a documented team choice
For each run, open the local `representatives.csv` to read the full articles. The three
examples per topic are nearest to that topic's embedding centroid **among nonreserved articles**.
The exported ID tables contain no full article bodies. Inspect small and low-coherence topics
as well as the largest ones; a high mean coherence alone does not establish usefulness.

Edit `Results/modern/selection.json`: fill `run_id`, `rationale`, and `selected_by`.
Explain the chosen balance of granularity, coherence, coverage, and article support.
No configuration is selected automatically, and legacy assistant-suggested scores are not reused.

In [5]:
for run_id in experiments.run_id:
    folder = OUT/'runs'/run_id
    inventory = a.topic_inventory(folder, folder, len(df))
    print(run_id, '— largest topics')
    display(inventory.query('topic_id >= 0').nlargest(5,'count')[['topic_id','keywords','count','coherence','representative_article_ids']])
    print('Lowest-coherence topics:')
    display(inventory.query('topic_id >= 0').nsmallest(3,'coherence')[['topic_id','keywords','count','coherence']])
    fig, ax = plt.subplots(figsize=(8,3))
    ax.bar(np.arange(len(inventory)), inventory.sort_values('count',ascending=False)['count'])
    ax.set(xlabel='Clusters ranked by size (includes unassigned)', ylabel='Articles', title=run_id)
    fig.tight_layout()
    for extension in ['png','pdf']: fig.savefig(folder/f'topic_sizes.{extension}',dpi=180)
    plt.close(fig)
selection = a.selection_record(OUT, df, experiments.run_id.tolist())
print('Selection pending: complete selection.json.' if selection is None else selection)

n15_c25_s42 — largest topics


,topic_id,keywords,count,coherence,representative_article_ids
1,0,"tesla, autopilot, musk, teslas, model, driver,...",545,0.695174,546160 | 1133411 | 567033
2,1,"waymo, waymos, krafcik, selfdriving, phoenix, ...",435,0.729731,546719 | 484629 | 546490
3,2,"levandowski, uber, waymo, secrets, ubers, waym...",342,0.873680,1341648 | 1450043 | 550330
4,3,"uber, ubers, kalanick, drivers, company, khosr...",167,0.614272,1231901 | 535564 | 567805
5,4,"apple, apples, project, titan, car, cook, ipho...",160,0.617204,1486861 | 2052862 | 1998813


Lowest-coherence topics:


,topic_id,keywords,count,coherence
47,46,"optimus, yard, remote, michigan, phantom, auto...",46,0.368599
56,55,"crunch, week, apple, daily, tech, app, inbox, ...",35,0.473523
37,36,"jobs, data, people, think, robots, things, wor...",57,0.476692


n15_c50_s42 — largest topics


,topic_id,keywords,count,coherence,representative_article_ids
1,0,"tesla, autopilot, musk, teslas, model, driver,...",545,0.678260,546160 | 1133411 | 567033
2,1,"levandowski, uber, waymo, ubers, secrets, trad...",458,0.819383,1341648 | 1450043 | 1921486
3,2,"waymo, selfdriving, waymos, krafcik, phoenix, ...",435,0.673349,546719 | 484629 | 546490
4,3,"think, like, people, right, going, know, just,...",395,0.695550,218059 | 1335182 | 530462
5,4,"cars, safety, vehicles, selfdriving, car, huma...",302,0.680481,150859 | 2675771 | 660755


... (15 outputs left out)

,topic_id,keywords,count,coherence,representative_article_ids
1,0,"tesla, autopilot, musk, teslas, model, said, d...",538,0.668395,546160 | 1133411 | 567033
2,1,"waymo, selfdriving, waymos, krafcik, phoenix, ...",438,0.633666,546719 | 484629 | 546490
3,2,"uber, levandowski, waymo, ubers, secrets, trad...",435,0.819383,1341648 | 1921486 | 1450043
4,3,"ford, fords, delivery, vehicles, said, autonom...",397,0.636431,524986 | 130247 | 2308500
5,4,"think, like, people, right, going, know, just,...",318,0.695550,218059 | 1335182 | 138257


Lowest-coherence topics:


,topic_id,keywords,count,coherence
21,20,"pm, week, mobility, scooters, new, electric, t...",124,0.324442
11,10,"new, mr, trump, percent, facebook, nyt, compan...",227,0.432506
18,17,"car, concept, cars, like, electric, design, fu...",145,0.516415


Selection pending: complete selection.json.


## Selected-configuration stability and labels
Once a team choice exists, run seeds 7 and 21 and compare against seed 42. ARI uses articles
assigned in both runs; always report its evaluated count/fraction and outlier agreement.

Fill each row of the selected run's `topic_labels.csv`: a short proposed label, supporting
evidence notes, and `confirmed=true` after team review. Existing edits are preserved.
Keyword labels remain explicitly provisional until confirmed. Independent review cannot start early.

In [6]:
selected_model = None
if selection is not None:
    selected_run = selection['run_id']
    selected_row = experiments.set_index('run_id').loc[selected_run]
    selected_model, _ = fit_configuration(int(selected_row.neighbors), int(selected_row.cluster_size))
    SELECTED = OUT/'selected'/selected_run
    SELECTED.mkdir(parents=True, exist_ok=True)
    write_json(SELECTED/'selection.json', selection)
    stability_rows = []
    baseline = pd.read_csv(OUT/'runs'/selected_run/'assignments.csv',dtype={'article_id':str})
    for seed in [7,21]:
        other, other_metrics = fit_configuration(int(selected_row.neighbors), int(selected_row.cluster_size), seed)
        other_assignments = pd.read_csv(OUT/'runs'/other_metrics['run_id']/'assignments.csv',dtype={'article_id':str})
        summary = ev.seed_publication_comparison(df, baseline, other_assignments, SELECTED/f'seed_{seed}')
        stability_rows.append(dict(seed=seed, **{k:v for k,v in summary.items() if not isinstance(v,list)}))
    pd.DataFrame(stability_rows).to_csv(SELECTED/'stability.csv',index=False)
    display(pd.DataFrame(stability_rows))
    run_manifest = json.loads((OUT/'runs'/selected_run/'manifest.json').read_text())
    label_identity = {'run_id':selected_run,'corpus_hash':corpus_hash(df),'run_identity_hash':digest(run_manifest['identity'])}
    labels = a.labels_table(OUT/'runs'/selected_run, SELECTED, label_identity)
    display(labels)
else:
    print('Seed reruns and final labels pending the team model choice; all grid evidence is available.')

Seed reruns and final labels pending the team model choice; all grid evidence is available.


In [7]:
SHOW_2D_MAP = False
if SHOW_2D_MAP and selected_model is not None:
    coordinates = UMAP(n_components=2,n_neighbors=15,min_dist=0,metric='cosine',random_state=42).fit_transform(embeddings)
    labels_2d = np.asarray(selected_model.topics_)
    fig, ax = plt.subplots(figsize=(10,7))
    ax.scatter(*coordinates[labels_2d==-1].T,c='gray',s=3,alpha=.15)
    ax.scatter(*coordinates[labels_2d>=0].T,c=labels_2d[labels_2d>=0],cmap='turbo',s=4,alpha=.6)
    ax.set_title('Article topics — 2D display only; not an evaluation space')
    ax.set_axis_off()
    fig.savefig(SELECTED/'document_topics.png',dpi=180,bbox_inches='tight')
    plt.show()

## Interpretation limits
Coherence uses the shared reference tokenizer, English stopwords, top-ten unigrams, and
110-token windows. Outliers are excluded from coherence/diversity/silhouette but reported separately.
Silhouette uses original 384-dimensional embeddings and at most 2,000 seeded assigned articles;
it does not measure separation in the visualization. Seed matching is descriptive: topics may
split or merge, and one-to-one matches do not establish semantic equivalence.